In [34]:
import json
import pandas as pd
from pathlib import Path

In [35]:
def load_and_process(path, model, strategy, split):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)

            pred = item.get("prediction", "").upper()
            gold = item.get("gold_label", "").upper()

            rows.append({
                "model": model,
                "strategy": strategy,
                "split": split,
                "prediction": pred,
                "gold": gold
            })

    return pd.DataFrame(rows)

In [36]:
paths = {
    "Llama3.1-8b": {
        "baseline": {
            "train": "../outputs/averitec_original_pipeline/2026-03-30_01-25-13/averitec_train.jsonl",
            "dev": "../outputs/averitec_original_pipeline/2026-03-30_01-25-13/averitec_dev.jsonl"
        },
        "iterative": {
            "train": "../outputs/averitec_iterative_pipeline/2026-03-30_01-27-11/averitec_train.jsonl",
            "dev": "../outputs/averitec_iterative_pipeline/2026-03-30_01-27-11/averitec_dev.jsonl"
        }
    },

    # adiciona os outros modelos aqui
}

In [37]:
dfs = []

for model, strategies in paths.items():
    for strategy, splits in strategies.items():
        for split, path in splits.items():
            df = load_and_process(path, model, strategy, split)
            dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

In [38]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(df):
    acc = accuracy_score(df["gold"], df["prediction"])
    pre, rec, f1, _ = precision_recall_fscore_support(
        df["gold"],
        df["prediction"],
        average="macro",
        zero_division=0
    )

    return acc, pre, rec, f1

In [39]:
results = []

for (model, strategy, split), group in df_all.groupby(["model", "strategy", "split"]):
    acc, pre, rec, f1 = compute_metrics(group)

    results.append({
        "model": model,
        "strategy": strategy,
        "split": split,
        "acc": acc,
        "pre": pre,
        "rec": rec,
        "f1": f1
    })

df_results = pd.DataFrame(results)

In [40]:
table = df_results.pivot_table(
    index="model",
    columns=["strategy", "split"],
    values=["acc", "pre", "rec", "f1"]
)

table = table.reorder_levels([1, 2, 0], axis=1)
table = table.sort_index(axis=1)

table = table.round(2)
table

strategy    baseline                                          iterative        \
split            dev                   train                        dev         
                 acc    f1   pre   rec   acc   f1   pre   rec       acc    f1   
model                                                                           
Llama3.1-8b     0.58  0.33  0.35  0.34  0.53  0.3  0.33  0.32      0.64  0.35   

strategy                                         
split                   train                    
              pre   rec   acc    f1   pre   rec  
model                                            
Llama3.1-8b  0.37  0.35  0.56  0.34  0.35  0.35